# Experiment E — 5-rep re-run (fresh session)

Everything else is already saved to Drive. This notebook only redoes
Experiment E at 5 reps instead of 3.

Run cells top to bottom. **~55 min total.**

| Cell | Does | Time |
|---|---|---|
| 1 | setup, restore the EXACT split from `_split.json` | 3 min |
| 2 | mask + crop helpers (prefers `shape_confound.py`) | instant |
| 3 | crop cache (segment once, reuse everywhere) | 15 min |
| 4 | DINO features **from the cache** | 3 min |
| 5 | 5-rep student training | 30 min |
| 6 | analysis + save | 1 min |

Reps 0–2 will reproduce the previous run exactly (seeds derive from `rep`).

In [ ]:
# ============================================================
# CELL 1 - setup + RESTORE the exact split (no re-hashing)
# ============================================================
!pip install -q timm opencv-python-headless

import os, json, time, shutil, hashlib, random
import numpy as np
import torch, torch.nn as nn
import torchvision.transforms as T
import torchvision.models as tvm
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import cv2
from tqdm.auto import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive'

CFG = {
    'RUN_TAG'   : 'revision_v1',
    'SEED'      : 42,
    'TEST_FRAC' : 0.30,
    'N_SHOT'    : 10,
    'N_REPEATS' : 5,          # <-- the point of this re-run
    'IMG_SIZE'  : 224,
    'CROP_PAD'  : 0.06,
    'DATA_ROOT' : '/content/drive/MyDrive/Tea Leaf Dataset',
    'LOCAL_DATA': '/tmp/tea_leaf_local',
    'OUT'       : '/content/drive/MyDrive/TeaLeaf_DINOv2_Study',
}
CFG['RESULTS_DIR'] = os.path.join(CFG['OUT'], 'results_' + CFG['RUN_TAG'])
os.makedirs(CFG['RESULTS_DIR'], exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
random.seed(CFG['SEED']); np.random.seed(CFG['SEED']); torch.manual_seed(CFG['SEED'])

def save_json(name, obj):
    obj = {'_cfg': CFG, '_saved': time.strftime('%Y-%m-%d %H:%M:%S'), **obj}
    p = os.path.join(CFG['RESULTS_DIR'], name)
    with open(p, 'w') as f:
        json.dump(obj, f, indent=2, default=float)
    print('saved ->', p); return p

# ---- copy dataset locally ----
def count_files(p):
    return sum(len(f) for _, _, f in os.walk(p)) if os.path.isdir(p) else 0

if count_files(CFG['LOCAL_DATA']) < count_files(CFG['DATA_ROOT']):
    from concurrent.futures import ThreadPoolExecutor
    tasks = []
    for root, _, files_ in os.walk(CFG['DATA_ROOT']):
        rel = os.path.relpath(root, CFG['DATA_ROOT'])
        dst = CFG['LOCAL_DATA'] if rel == '.' else os.path.join(CFG['LOCAL_DATA'], rel)
        os.makedirs(dst, exist_ok=True)
        for f in files_:
            s, d = os.path.join(root, f), os.path.join(dst, f)
            if not os.path.exists(d):
                tasks.append((s, d))
    def _cp(t):
        try:
            shutil.copyfile(t[0], t[1]); return True
        except Exception:
            return False
    with ThreadPoolExecutor(max_workers=32) as ex:
        ok = list(tqdm(ex.map(_cp, tasks), total=len(tasks), desc='copy'))
    print('copied', sum(ok), '/', len(tasks))
else:
    print('local cache complete:', count_files(CFG['LOCAL_DATA']), 'files')

classes = sorted(d for d in os.listdir(CFG['LOCAL_DATA'])
                 if os.path.isdir(os.path.join(CFG['LOCAL_DATA'], d)))
CLASS_TO_IDX = {c: i for i, c in enumerate(classes)}
healthy_idx = [CLASS_TO_IDX[c] for c in classes if 'healthy' in c.lower()]

IMAGES = []
for c in classes:
    d = os.path.join(CFG['LOCAL_DATA'], c)
    for fn in sorted(os.listdir(d)):
        if fn.lower().endswith(('.jpg', '.jpeg', '.png')):
            IMAGES.append({'path': os.path.join(d, fn), 'cls': c,
                           'y': CLASS_TO_IDX[c],
                           'diseased': 0 if CLASS_TO_IDX[c] in healthy_idx else 1})
KEEP = IMAGES
print('indexed:', len(KEEP), '| classes:', classes)

# ---- restore the split written by the previous session ----
sp = os.path.join(CFG['RESULTS_DIR'], '_split.json')
assert os.path.exists(sp), 'MISSING %s - cannot restore the split. Re-run the ' \
                           'audit cell from the original notebook first.' % sp
SPLIT = json.load(open(sp))
test_paths = set(SPLIT['test_paths'])
TRAIN = [r for r in KEEP if r['path'] not in test_paths]
TEST  = [r for r in KEEP if r['path'] in test_paths]

assert len(TRAIN) == SPLIT['n_train'] and len(TEST) == SPLIT['n_test'], \
    'split restore mismatch: got %d/%d, expected %d/%d' % (
        len(TRAIN), len(TEST), SPLIT['n_train'], SPLIT['n_test'])
print('\\nSPLIT RESTORED from %s' % SPLIT.get('audit_source', 'unknown'))
print('  train %d | test %d | test diseased %.1f%%'
      % (len(TRAIN), len(TEST), 100 * np.mean([r['diseased'] for r in TEST])))

In [ ]:
# ============================================================
# CELL 2 - mask + crop (prefers your original module)
# ============================================================
def try_import(name):
    src = os.path.join(CFG['OUT'], name + '.py')
    if os.path.exists(src):
        shutil.copy(src, '/content/' + name + '.py')
        try:
            m = __import__(name); print('  using ORIGINAL module:', name + '.py'); return m
        except Exception as e:
            print('  ! import failed', name, e)
    else:
        print('  not on Drive:', name + '.py')
    return None

sc = try_import('shape_confound')

def _leaf_mask_fallback(rgb):
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    v, s = hsv[:, :, 2], hsv[:, :, 1]
    _, mv = cv2.threshold(v, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    _, ms = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m = ((mv > 0) | (ms > 0)).astype(np.uint8)
    n, lab, stats, _ = cv2.connectedComponentsWithStats(m, 8)
    if n <= 1:
        return np.ones(rgb.shape[:2], np.uint8)
    k = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    m = (lab == k).astype(np.uint8)
    ff = m.copy(); h, w = m.shape
    cv2.floodFill(ff, np.zeros((h + 2, w + 2), np.uint8), (0, 0), 1)
    return (m | (1 - ff)).astype(np.uint8)

def leaf_mask(rgb):
    if sc is not None and hasattr(sc, 'leaf_mask'):
        try:
            m = sc.leaf_mask(Image.fromarray(rgb))
            if m is not None:
                m = np.asarray(m)
                if m.max() > 1:
                    m = (m > 127).astype(np.uint8)
                m = m.astype(np.uint8)
                if m.shape == rgb.shape[:2]:
                    return m
        except Exception:
            pass
    return _leaf_mask_fallback(rgb)

def crop_to_leaf(rgb, m, pad=None):
    pad = CFG['CROP_PAD'] if pad is None else pad
    ys, xs = np.where(m > 0)
    if len(xs) < 16:
        return Image.fromarray(rgb).resize((CFG['IMG_SIZE'],) * 2)
    x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()
    bw, bh = x1 - x0 + 1, y1 - y0 + 1
    px, py = int(pad * bw), int(pad * bh)
    H, W = rgb.shape[:2]
    x0 = max(0, x0 - px); x1 = min(W - 1, x1 + px)
    y0 = max(0, y0 - py); y1 = min(H - 1, y1 + py)
    return Image.fromarray(rgb[y0:y1 + 1, x0:x1 + 1]).resize((CFG['IMG_SIZE'],) * 2)

print('mask source:',
      'shape_confound.leaf_mask' if (sc and hasattr(sc, 'leaf_mask')) else 'fallback')

In [ ]:
# ============================================================
# CELL 3 - crop cache  (segment ONCE, reuse for everything)
# ============================================================
CACHE = '/content/crop_cache'
os.makedirs(CACHE, exist_ok=True)

def cache_path(rec):
    return os.path.join(CACHE, hashlib.md5(rec['path'].encode()).hexdigest() + '.jpg')

todo = [r for r in KEEP if not os.path.exists(cache_path(r))]
print('to build:', len(todo), '| already cached:', len(KEEP) - len(todo))
for rec in tqdm(todo, desc='crop cache'):
    rgb = np.array(Image.open(rec['path']).convert('RGB'))
    crop_to_leaf(rgb, leaf_mask(rgb)).save(cache_path(rec), quality=95)
print('cache size:', len(os.listdir(CACHE)))

TO_T = T.Compose([T.ToTensor(),
                  T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

class CachedDS(Dataset):
    # reads the cached crop; no segmentation at train time
    def __init__(self, recs, labels=None, train=False):
        self.r = recs
        self.l = np.zeros(len(recs), int) if labels is None else np.asarray(labels)
        self.train = train
        self.aug = T.Compose([T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                              T.RandomRotation(20), T.ColorJitter(.2, .2, .2)])
    def __len__(self):
        return len(self.r)
    def __getitem__(self, i):
        im = Image.open(cache_path(self.r[i])).convert('RGB')
        if self.train:
            im = self.aug(im)
        return TO_T(im), int(self.l[i]), self.r[i]['y'], self.r[i]['diseased']

In [ ]:
# ============================================================
# CELL 4 - DINOv1 features from the cache (cached to Drive)
# ============================================================
import timm

FEAT = os.path.join(CFG['RESULTS_DIR'], 'features_dinov1_crop.npz')

def load_dinov1():
    p = os.path.join(CFG['OUT'], 'checkpoints', 'dino_vits16.pth')
    if not os.path.exists(p):
        os.makedirs(os.path.dirname(p), exist_ok=True)
        os.system('wget -q -O %s https://dl.fbaipublicfiles.com/dino/'
                  'dino_deitsmall16_pretrain/dino_deitsmall16_pretrain.pth' % p)
    m = timm.create_model('vit_small_patch16_224', pretrained=False, num_classes=0)
    ck = torch.load(p, map_location='cpu')
    sd = {k.replace('module.', '').replace('backbone.', '').replace('encoder.', ''): v
          for k, v in ck.items() if 'head' not in k}
    m.load_state_dict(sd, strict=False)
    return m.eval().to(device)

@torch.no_grad()
def extract(model, recs, desc='feat'):
    dl = DataLoader(CachedDS(recs), batch_size=64, num_workers=4, shuffle=False)
    F, Y, D = [], [], []
    for x, _, y, d in tqdm(dl, desc=desc):
        f = torch.nn.functional.normalize(model(x.to(device)), dim=1)
        F.append(f.cpu().numpy()); Y.append(y.numpy()); D.append(d.numpy())
    return np.concatenate(F), np.concatenate(Y), np.concatenate(D)

if os.path.exists(FEAT):
    z = np.load(FEAT)
    Xtr, ytr, dtr, Xte, yte, dte = (z['Xtr'], z['ytr'], z['dtr'],
                                    z['Xte'], z['yte'], z['dte'])
    print('features loaded from Drive:', Xtr.shape, Xte.shape)
else:
    enc = load_dinov1()
    Xtr, ytr, dtr = extract(enc, TRAIN, 'train')
    Xte, yte, dte = extract(enc, TEST,  'test')
    np.savez_compressed(FEAT, Xtr=Xtr, ytr=ytr, dtr=dtr, Xte=Xte, yte=yte, dte=dte)
    print('features saved ->', FEAT)
    del enc; torch.cuda.empty_cache()


def fit_probe(X, y, idx):
    p = LogisticRegression(max_iter=3000, C=1.0)   # lbfgs -> multinomial by default
    p.fit(X[idx], y[idx]); return p

def binary_score(probe, X):
    P = probe.predict_proba(X)
    cols = [i for i, c in enumerate(probe.classes_) if c in healthy_idx]
    return 1.0 - P[:, cols].sum(1)

# sanity: probe AUC must match A_mask_perturbation.json 'clean' runs
chk = []
for rep in range(3):
    rng = np.random.default_rng(CFG['SEED'] * 1000 + rep)
    idx = np.concatenate([rng.choice(np.where(ytr == c)[0], CFG['N_SHOT'], replace=False)
                          for c in range(len(classes))])
    chk.append(roc_auc_score(dte, binary_score(fit_probe(Xtr, ytr, idx), Xte)))
print('\\nprobe AUC reps 0-2 :', np.round(chk, 4).tolist())
print('expected (prev run) : [0.9767, 0.9825, 0.9868]')

In [ ]:
# ============================================================
# CELL 5 - EXPERIMENT E at 5 reps   (~30 min)
# ============================================================
def train_student(recs, labels, epochs=30, bs=64):
    m = tvm.mobilenet_v3_small(weights=tvm.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
    m.classifier[3] = nn.Linear(m.classifier[3].in_features, 2)
    m = m.to(device)
    dl = DataLoader(CachedDS(recs, labels, train=True), batch_size=bs,
                    shuffle=True, num_workers=4)
    opt = optim.Adam(m.parameters(), lr=1e-3)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    lf = nn.CrossEntropyLoss()
    for _ in range(epochs):
        m.train()
        for x, y, _, _ in dl:
            opt.zero_grad(); lf(m(x.to(device)), y.to(device)).backward(); opt.step()
        sch.step()
    return m

@torch.no_grad()
def student_scores(m, recs):
    m.eval()
    dl = DataLoader(CachedDS(recs), batch_size=128, num_workers=4)
    return np.concatenate([torch.softmax(m(x.to(device)), 1)[:, 1].cpu().numpy()
                           for x, _, _, _ in dl])

S = {'labels_only': [], 'probe_distil': []}
probe_auc, pseudo_acc, taus = [], [], []

for rep in tqdm(range(CFG['N_REPEATS']), desc='reps'):
    rng = np.random.default_rng(CFG['SEED'] * 1000 + rep)
    idx = np.concatenate([rng.choice(np.where(ytr == c)[0], CFG['N_SHOT'], replace=False)
                          for c in range(len(classes))])
    seed_recs, seed_bin = [TRAIN[i] for i in idx], dtr[idx]

    S['labels_only'].append(student_scores(train_student(seed_recs, seed_bin), TEST))

    probe = fit_probe(Xtr, ytr, idx)
    probe_auc.append(float(roc_auc_score(dte, binary_score(probe, Xte))))
    s_tr = binary_score(probe, Xtr)
    tau  = float(np.quantile(s_tr, 1.0 - float(dtr.mean())))   # Sec 3.4
    pseudo = (s_tr > tau).astype(int); pseudo[idx] = seed_bin
    taus.append(tau); pseudo_acc.append(float((pseudo == dtr).mean()))
    S['probe_distil'].append(student_scores(train_student(TRAIN, pseudo), TEST))

    print('rep %d | tau %.3f | pseudo-acc %.3f | probe %.4f | labels %.4f | distil %.4f'
          % (rep, tau, pseudo_acc[-1], probe_auc[-1],
             roc_auc_score(dte, S['labels_only'][-1]),
             roc_auc_score(dte, S['probe_distil'][-1])))

np.savez_compressed(os.path.join(CFG['RESULTS_DIR'], 'E_scores_5rep.npz'),
                    labels_only=np.array(S['labels_only']),
                    probe_distil=np.array(S['probe_distil']), dte=dte)
print('\\nraw per-image scores saved (so this never has to be re-run)')

In [ ]:
# ============================================================
# CELL 6 - analysis + save
# ============================================================
per = {}
for arm in ('labels_only', 'probe_distil'):
    per[arm] = [float(roc_auc_score(dte, s)) for s in S[arm]]
    print('%-13s per-rep %s\\n%14s single %.4f +/- %.4f | ensemble %.4f'
          % (arm, np.round(per[arm], 4).tolist(), '',
             np.mean(per[arm]), np.std(per[arm], ddof=1),
             roc_auc_score(dte, np.mean(S[arm], 0))))

lo = np.array(per['labels_only']); pd_ = np.array(per['probe_distil'])
pr = np.array(probe_auc)
d  = pd_ - lo
rl, rp = lo.max() - lo.min(), pd_.max() - pd_.min()

print('\\nprobe (teacher)   : %.4f +/- %.4f' % (pr.mean(), pr.std(ddof=1)))
print('paired by rep     : mean %+.4f | per-rep %s | wins %d/%d'
      % (d.mean(), np.round(d, 4).tolist(), int((d > 0).sum()), len(d)))
print('range             : labels %.4f | distil %.4f  -> %.1fx narrower'
      % (rl, rp, rl / rp))
print('sd                : labels %.4f | distil %.4f  -> %.1fx'
      % (lo.std(ddof=1), pd_.std(ddof=1), lo.std(ddof=1) / pd_.std(ddof=1)))
print('distil - teacher  : %+.4f' % (pd_.mean() - pr.mean()))

print('\\n' + '=' * 70)
print('%-32s %-12s %s' % ('', 'published', 'this run (%d reps)' % len(d)))
print('-' * 70)
print('%-32s %-12s %.4f +/- %.4f' % ('Supervised-only, 70 labels', '0.958',
                                     lo.mean(), lo.std(ddof=1)))
print('%-32s %-12s %.4f +/- %.4f' % ('Probe + distillation (ours)', '0.973',
                                     pd_.mean(), pd_.std(ddof=1)))
print('%-32s %-12s %.4f +/- %.4f' % ('Probe alone (teacher)', '0.984',
                                     pr.mean(), pr.std(ddof=1)))
print('=' * 70)
print('SEC 4.12 NUMBERS -> gain %+.4f on %d/%d seeds; labels span %.3f-%.3f, '
      'distil %.3f-%.3f; %.1fx spread reduction'
      % (d.mean(), int((d > 0).sum()), len(d), lo.min(), lo.max(),
         pd_.min(), pd_.max(), rl / rp))

save_json('E_distillation_5rep.json', {
    'threshold_rule': 'tau = quantile(scores, healthy_prevalence)  [Sec 3.4]',
    'n_test': len(TEST), 'n_reps': int(CFG['N_REPEATS']),
    'per_rep': per, 'probe_auc_per_rep': probe_auc,
    'pseudo_label_acc': pseudo_acc, 'taus': taus,
    'single_model': {k: {'mean': float(np.mean(v)), 'std': float(np.std(v, ddof=1))}
                     for k, v in per.items()},
    'ensemble': {k: float(roc_auc_score(dte, np.mean(S[k], 0))) for k in per},
    'paired_gain_mean': float(d.mean()), 'paired_gain_per_rep': d.tolist(),
    'wins': int((d > 0).sum()),
    'range': {'labels_only': float(rl), 'probe_distil': float(rp),
              'ratio': float(rl / rp)},
    'sd_ratio': float(lo.std(ddof=1) / pd_.std(ddof=1)),
    'distil_minus_teacher': float(pd_.mean() - pr.mean()),
})

# Figure regeneration

Only two figures need work.

| Figure | Status | Cell |
|---|---|---|
| Fig. 1, 2, 3, 5, 6, 7 | **no change** — caption fixes only, already in the .tex | — |
| **Fig. 4** (cross-dataset) | must change either way | 2 (withdraw) or 5 (SAM) |
| **attention maps** | only if any came from `TEA_LEAF_FIGURES.ipynb` | 6 |

### Two routes for Figure 4

**Route A — withdraw the generalisation claim.** Cell 2. No GPU, ~10 seconds.
Redraws Fig. 4 with a fourth panel showing that the Otsu mask covers the whole
frame on every external dataset, which is why the other three panels are flat.

**Route B — re-segment with SAM and re-run the diagnostic.** Cells 3–5, ~40 min.
If the confound signature is still absent under masks that actually find a leaf,
RQ3 survives and is far better supported than before.

Run Cell 2 regardless — you need it as the fallback, and it tells you how bad
the segmentation problem is before you spend 40 minutes on SAM.

In [ ]:
# ============================================================
# CELL 0 - setup
# ============================================================
!pip install -q timm opencv-python-headless

import os, json, glob, hashlib, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import cv2, torch
from PIL import Image
from tqdm.auto import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

CFG = {
    'SEED'      : 42,
    'OUT'       : '/content/drive/MyDrive/TeaLeaf_DINOv2_Study',
    'LOCAL_DATA': '/tmp/tea_leaf_local',
    'N_PER_SET' : 400,        # images sampled per external dataset
    'GRID'      : 14,         # 224/16 for ViT-S/16
    'TOPK'      : int(0.1 * 14 * 14),   # kappa = 19
}
CFG['FIG'] = os.path.join(CFG['OUT'], 'figures_revision')
os.makedirs(CFG['FIG'], exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

mpl.rcParams.update({'font.size': 9, 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.dpi': 130})
OURS, EXT, WARN = '#E8853A', '#BFBFBF', '#C0392B'
print('device:', device, '| figures ->', CFG['FIG'])

In [ ]:
# ============================================================
# CELLS 0+1 COMBINED - run this, then Cell 2 onwards
# ============================================================
!pip install -q timm opencv-python-headless

import os, json, glob, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import cv2, torch
from PIL import Image
from tqdm.auto import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

CFG = {
    'SEED'      : 42,
    'OUT'       : '/content/drive/MyDrive/TeaLeaf_DINOv2_Study',
    'LOCAL_DATA': '/tmp/tea_leaf_local',
    'TEA_DRIVE' : '/content/drive/MyDrive/Tea Leaf Dataset',
    'N_PER_SET' : 400,
    'GRID'      : 14,
    'TOPK'      : int(0.1 * 14 * 14),
}
CFG['FIG'] = os.path.join(CFG['OUT'], 'figures_revision')
os.makedirs(CFG['FIG'], exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

mpl.rcParams.update({'font.size': 9, 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.dpi': 130})
OURS, EXT, WARN = '#E8853A', '#BFBFBF', '#C0392B'
print('device:', device, '| figures ->', CFG['FIG'])

# ---- locate datasets ----
KEYS = {'PlantVillage': ['plantvillage', 'plant_village'],
        'PlantDoc': ['plantdoc', 'plant_doc'],
        'RoCoLe': ['rocole', 'robusta'],
        'RiceLeafDiseaseBD': ['riceleaf', 'rice_leaf', 'riceleafbd']}

def discover(base='/content/drive/MyDrive', max_depth=4):
    found, bd = {}, base.rstrip('/').count('/')
    for root, dirs, _ in os.walk(base):
        if root.count('/') - bd > max_depth:
            dirs[:] = []; continue
        low = os.path.basename(root).lower().replace(' ', '')
        for n, pats in KEYS.items():
            if n not in found and any(p in low for p in pats):
                found[n] = root
    return found

DATASETS = discover()
tea = CFG['LOCAL_DATA'] if (os.path.isdir(CFG['LOCAL_DATA'])
                            and os.listdir(CFG['LOCAL_DATA'])) else CFG['TEA_DRIVE']
DATASETS['TeaLeafDiseaseBD'] = tea

print('\ndataset roots:')
for k, v in DATASETS.items():
    print('  %-20s %s' % (k, v))

print('\nclass folders:')
for k, v in DATASETS.items():
    if os.path.isdir(v):
        subs = sorted(d for d in os.listdir(v) if os.path.isdir(os.path.join(v, d)))
        print('  %-20s %s' % (k, subs[:10]))

# ---- healthy / diseased labelling ----
HEALTHY_TOKENS = ['healthy', 'health', 'fresh', 'normal']

def is_healthy(path):
    p = path.lower()
    return any(t in p for t in HEALTHY_TOKENS)

def sample_set(root, n, seed=CFG['SEED']):
    paths = []
    for r, _, fs in os.walk(root):
        for f in fs:
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                paths.append(os.path.join(r, f))
    h = [p for p in paths if is_healthy(p)]
    d = [p for p in paths if not is_healthy(p)]
    rng = np.random.default_rng(seed)
    k = min(n // 2, len(h), len(d))
    if k == 0:
        return [], np.array([])
    sel = ([h[i] for i in rng.choice(len(h), k, replace=False)] +
           [d[i] for i in rng.choice(len(d), k, replace=False)])
    return sel, np.array([0] * k + [1] * k)

SAMPLES = {}
print('\nsamples:')
for name, root in DATASETS.items():
    if not os.path.isdir(root):
        print('  %-20s MISSING' % name); continue
    p, y = sample_set(root, CFG['N_PER_SET'])
    if len(p) == 0:
        print('  %-20s SKIP - no folder matched %s' % (name, HEALTHY_TOKENS)); continue
    SAMPLES[name] = (p, y)
    print('  %-20s n=%4d  (%d healthy / %d diseased)'
          % (name, len(p), (y == 0).sum(), (y == 1).sum()))

assert 'TeaLeafDiseaseBD' in SAMPLES, 'your own dataset must be in the comparison'
print('\nready:', list(SAMPLES))

In [ ]:
# ============================================================
# CELL 1 - locate datasets + healthy/diseased labelling
# ============================================================
KEYS = {'PlantVillage': ['plantvillage', 'plant_village'],
        'PlantDoc': ['plantdoc', 'plant_doc'],
        'RoCoLe': ['rocole', 'robusta'],
        'RiceLeafDiseaseBD': ['riceleaf', 'rice_leaf', 'riceleafbd']}

def discover(base='/content/drive/MyDrive', max_depth=4):
    found, bd = {}, base.rstrip('/').count('/')
    for root, dirs, _ in os.walk(base):
        if root.count('/') - bd > max_depth:
            dirs[:] = []; continue
        low = os.path.basename(root).lower().replace(' ', '')
        for n, pats in KEYS.items():
            if n not in found and any(p in low for p in pats):
                found[n] = root
    return found

DATASETS = discover()
DATASETS['TeaLeafDiseaseBD'] = CFG['LOCAL_DATA']
for k, v in DATASETS.items():
    print('%-20s %s' % (k, v))
missing = [k for k in KEYS if k not in DATASETS]
if missing:
    print('\nNOT FOUND: %s  -> set DATASETS[name] = "/path" manually' % ', '.join(missing))

HEALTHY_TOKENS = ['healthy', 'health', 'fresh', 'normal']

def is_healthy(path):
    p = path.lower()
    return any(t in p for t in HEALTHY_TOKENS)

def sample_set(root, n, seed=CFG['SEED']):
    # balanced healthy / diseased sample, labelled from the folder name
    paths = []
    for r, _, fs in os.walk(root):
        for f in fs:
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                paths.append(os.path.join(r, f))
    h = [p for p in paths if is_healthy(p)]
    d = [p for p in paths if not is_healthy(p)]
    rng = np.random.default_rng(seed)
    k = min(n // 2, len(h), len(d))
    if k == 0:
        return [], np.array([])
    sel = ([h[i] for i in rng.choice(len(h), k, replace=False)] +
           [d[i] for i in rng.choice(len(d), k, replace=False)])
    return sel, np.array([0] * k + [1] * k)

SAMPLES = {}
for name, root in DATASETS.items():
    if not os.path.isdir(root):
        continue
    p, y = sample_set(root, CFG['N_PER_SET'])
    if len(p) == 0:
        print('SKIP %s - could not identify healthy images by folder name' % name)
        continue
    SAMPLES[name] = (p, y)
    print('%-20s n=%4d  (%d healthy / %d diseased)' % (name, len(p), (y == 0).sum(), (y == 1).sum()))

In [ ]:
# ============================================================
# CELL 2 - FIGURE 4, ROUTE A  (withdraw the claim)   ~10 s, no GPU
# ============================================================
# Panels a-c are the published numbers, unchanged. Panel d is new and
# explains why b and c are flat: the Otsu mask is the whole frame on
# every external dataset, so the shape descriptors are near-constant
# and "within-leaf" equals "whole image".

PUB = {   # published diagnostic values (Tea_Leaf_Re__1_ cross-dataset cell)
    'TeaLeafDiseaseBD' : dict(ratio=1.94, shape=0.834, delta=+0.385),
    'PlantDoc'         : dict(ratio=1.07, shape=0.600, delta=-0.013),
    'PlantVillage'     : dict(ratio=1.02, shape=0.574, delta=+0.037),
    'RiceLeafDiseaseBD': dict(ratio=1.06, shape=0.623, delta=+0.011),
    'RoCoLe'           : dict(ratio=0.92, shape=0.572, delta=-0.023),
}
AREA = {  # C_external_masks.json - median Otsu mask area
    'TeaLeafDiseaseBD': 0.095, 'PlantDoc': 1.000, 'PlantVillage': 1.000,
    'RiceLeafDiseaseBD': 1.000, 'RoCoLe': 1.000,
}
ORDER = ['TeaLeafDiseaseBD', 'PlantDoc', 'PlantVillage', 'RiceLeafDiseaseBD', 'RoCoLe']
SHORT = ['TeaLeaf\nDiseaseBD', 'PlantDoc\n(field)', 'Plant\nVillage',
         'RiceLeaf\nDiseaseBD', 'RoCoLe']

fig, ax = plt.subplots(1, 4, figsize=(13.5, 3.4))
cols = [OURS] + [EXT] * 4
x = np.arange(5)

panels = [
    ('a', 'ratio', 'Leaf-area ratio\n(healthy / diseased)', None,  (0, 2.15), '%.2f'),
    ('b', 'shape', 'C1: leaf geometry alone\n(AUC)',        0.70,  (0.5, 0.92), '%.2f'),
    ('c', 'delta', 'C2: attention collapse\nrestricted to the leaf', 0.15, (-0.08, 0.46), '%+.2f'),
]
for i, (tag, key, title, thr, ylim, fmt) in enumerate(panels):
    a = ax[i]
    v = [PUB[k][key] for k in ORDER]
    a.bar(x, v, color=cols, edgecolor='none', width=.68)
    for xi, vi in zip(x, v):
        a.text(xi, vi + (ylim[1] - ylim[0]) * .025, fmt % vi, ha='center',
               fontsize=7.5, fontweight='bold' if xi == 0 else 'normal')
    if thr is not None:
        a.axhline(thr, ls='--', lw=.9, color='#4A78B5')
        a.text(4.45, thr, ' confound\n threshold', fontsize=6.5,
               color='#4A78B5', va='bottom', ha='right')
    if key == 'ratio':
        a.axhline(1.0, ls=':', lw=.9, color='#888')
    a.set_ylim(*ylim); a.set_xticks(x)
    a.set_xticklabels(SHORT, fontsize=6.5)
    a.set_title(title, fontsize=8.5)
    a.text(-.16, 1.14, tag, transform=a.transAxes, fontsize=12, fontweight='bold')

# --- panel d: the segmentation-validity check ---
a = ax[3]
v = [AREA[k] for k in ORDER]
a.bar(x, v, color=[OURS] + [WARN] * 4, edgecolor='none', width=.68)
for xi, vi in zip(x, v):
    a.text(xi, vi + .025, '%.2f' % vi, ha='center', fontsize=7.5,
           fontweight='bold' if xi == 0 else 'normal')
a.axhline(1.0, ls='--', lw=.9, color=WARN)
a.text(4.45, 1.0, ' whole frame', fontsize=6.5, color=WARN, va='bottom', ha='right')
a.set_ylim(0, 1.18); a.set_xticks(x); a.set_xticklabels(SHORT, fontsize=6.5)
a.set_title('Median segmented area\n(fraction of frame)', fontsize=8.5)
a.text(-.16, 1.14, 'd', transform=a.transAxes, fontsize=12, fontweight='bold')

fig.suptitle('The diagnostic requires a mask that finds the object', fontsize=10.5, y=1.04)
fig.tight_layout()
p = os.path.join(CFG['FIG'], 'fig4_routeA_withdraw.png')
fig.savefig(p, dpi=300, bbox_inches='tight'); plt.show()
print('saved ->', p)
CAPTION_A = [
 'CAPTION (Route A):',
 '  The cross-dataset diagnostic and its scope. (a-c) The three controls applied',
 '  to five datasets; our dataset is an outlier on every axis. (d) Median area of',
 '  the Otsu segmentation as a fraction of the frame. On all four external',
 '  datasets the mask covers essentially the whole image, because the threshold',
 '  rule of Section 3.3 relies on the dark studio backdrop and does not transfer',
 '  to light or textured backgrounds. Panels b and c are therefore uninformative',
 '  for those datasets: near-constant shape descriptors force a low shape-only',
 '  AUC, and a whole-frame mask makes the within-object score identical to the',
 '  naive one. We report the comparison for completeness but do not claim on this',
 '  evidence that the confound is absent elsewhere.']
print(chr(10).join(CAPTION_A))

In [ ]:
# ---- fix: point TeaLeafDiseaseBD at Drive, and verify every dataset ----
TEA_DRIVE = '/content/drive/MyDrive/Tea Leaf Dataset'

if not (os.path.isdir(CFG['LOCAL_DATA']) and os.listdir(CFG['LOCAL_DATA'])):
    assert os.path.isdir(TEA_DRIVE), 'set TEA_DRIVE to your tea dataset folder'
    DATASETS['TeaLeafDiseaseBD'] = TEA_DRIVE
    print('TeaLeafDiseaseBD -> Drive (reading 400 sampled images directly)')

SAMPLES = {}
for name, root in DATASETS.items():
    if not os.path.isdir(root):
        print('%-20s MISSING: %s' % (name, root)); continue
    p, y = sample_set(root, CFG['N_PER_SET'])
    if len(p) == 0:
        print('%-20s SKIP - no folder matched %s' % (name, HEALTHY_TOKENS)); continue
    SAMPLES[name] = (p, y)
    print('%-20s n=%4d  (%d healthy / %d diseased)'
          % (name, len(p), (y == 0).sum(), (y == 1).sum()))

assert 'TeaLeafDiseaseBD' in SAMPLES, 'your own dataset must be in the comparison'
print('\nready:', list(SAMPLES))

In [ ]:
# ============================================================
# CELL 3 - ROUTE B step 1: SAM masks for the external sets  (~15 min)
# ============================================================
RUN_SAM = True

if RUN_SAM:
    !pip install -q git+https://github.com/facebookresearch/segment-anything.git
    CKPT = '/content/sam_vit_b_01ec64.pth'
    if not os.path.exists(CKPT):
        !wget -q -O {CKPT} https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
    from segment_anything import sam_model_registry, SamPredictor

    sam = sam_model_registry['vit_b'](checkpoint=CKPT).to(device)
    predictor = SamPredictor(sam)

    def sam_mask(rgb):
        # centre-point prompt: the leaf is the subject of the photograph
        H, W = rgb.shape[:2]
        predictor.set_image(rgb)
        pts = np.array([[W // 2, H // 2]])
        lbl = np.array([1])
        masks, scores, _ = predictor.predict(point_coords=pts, point_labels=lbl,
                                             multimask_output=True)
        # reject degenerate picks: <2% or >85% of the frame
        cand = [(s, m) for s, m in zip(scores, masks)
                if 0.02 < m.mean() < 0.85]
        if not cand:
            return None
        return max(cand, key=lambda t: t[0])[1].astype(np.uint8)

    SAM_MASKS = {}
    for name, (paths, y) in SAMPLES.items():
        ms, keep = [], []
        for i, p in enumerate(tqdm(paths, desc='SAM ' + name)):
            rgb = np.array(Image.open(p).convert('RGB'))
            if max(rgb.shape[:2]) > 1024:
                sc = 1024 / max(rgb.shape[:2])
                rgb = cv2.resize(rgb, None, fx=sc, fy=sc)
            m = sam_mask(rgb)
            if m is not None:
                ms.append(m); keep.append(i)
        SAM_MASKS[name] = (keep, ms)
        area = np.array([m.mean() for m in ms])
        print('  %-20s kept %d/%d | median area %.3f | frac>0.5 %.2f'
              % (name, len(ms), len(paths), np.median(area), (area > .5).mean()))

    del sam, predictor; torch.cuda.empty_cache()
    print('\nSanity: median area should now be ~0.1-0.5, NOT 1.0.')
    print('If it is still near 1.0, SAM is not finding a leaf either and')
    print('Route A is the honest option.')

In [ ]:
# ============================================================
# CELL 4 - ROUTE B step 2: re-run the diagnostic on SAM masks (~20 min)
# ============================================================
if RUN_SAM:
    import timm
    _ATTN = {}

    def load_dinov1():
        p = os.path.join(CFG['OUT'], 'checkpoints', 'dino_vits16.pth')
        if not os.path.exists(p):
            os.makedirs(os.path.dirname(p), exist_ok=True)
            os.system('wget -q -O %s https://dl.fbaipublicfiles.com/dino/'
                      'dino_deitsmall16_pretrain/dino_deitsmall16_pretrain.pth' % p)
        m = timm.create_model('vit_small_patch16_224', pretrained=False, num_classes=0)
        ck = torch.load(p, map_location='cpu')
        sd = {k.replace('module.', '').replace('backbone.', ''): v
              for k, v in ck.items() if 'head' not in k}
        m.load_state_dict(sd, strict=False)
        blk = m.blocks[-1].attn
        def fwd(x, **kw):
            B, N, C = x.shape
            qkv = blk.qkv(x).reshape(B, N, 3, blk.num_heads, C // blk.num_heads).permute(2, 0, 3, 1, 4)
            q, k, v = qkv.unbind(0)
            sc = (C // blk.num_heads) ** -0.5
            _ATTN['cls'] = (((q[:, :, 0:1] @ k.transpose(-2, -1)) * sc)
                            .softmax(-1)[:, :, 0, 1:].detach().cpu())
            x = ((((q @ k.transpose(-2, -1)) * sc).softmax(-1)) @ v).transpose(1, 2).reshape(B, N, C)
            return blk.proj_drop(blk.proj(x))
        blk.forward = fwd
        return m.eval().to(device)

    enc = load_dinov1()
    MEAN = np.array([0.485, 0.456, 0.406]); STD = np.array([0.229, 0.224, 0.225])

    def shape_desc(m):
        H, W = m.shape; area = float(m.sum())
        if area < 16: return [0.] * 7
        cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        c = max(cnts, key=cv2.contourArea)
        peri = float(cv2.arcLength(c, True))
        hull = float(cv2.contourArea(cv2.convexHull(c))) + 1e-6
        x, y, bw, bh = cv2.boundingRect(c)
        ecc = 0.
        if len(c) >= 5:
            (_, _), (MA, ma), _ = cv2.fitEllipse(c)
            MA, ma = max(MA, ma), min(MA, ma)
            ecc = float(np.sqrt(max(0., 1 - (ma / (MA + 1e-6)) ** 2)))
        return [area / (H * W), bw / (bh + 1e-6), area / hull,
                4 * np.pi * area / (peri ** 2 + 1e-6), peri / (np.sqrt(area) + 1e-6),
                ecc, area / (bw * bh + 1e-6)]

    @torch.no_grad()
    def attn_scores(rgb, m):
        g = CFG['GRID']
        x = cv2.resize(rgb, (224, 224)).astype(np.float32) / 255.
        x = ((x - MEAN) / STD).transpose(2, 0, 1)[None]
        _ = enc(torch.tensor(x, dtype=torch.float32, device=device))
        a = _ATTN['cls'][0].numpy().mean(0)              # mean over heads, (196,)
        a = a / (a.sum() + 1e-12)
        naive = np.sort(a)[-CFG['TOPK']:].sum()
        mg = (cv2.resize(m.astype(np.float32), (g, g), interpolation=cv2.INTER_AREA) > .5).ravel()
        if mg.sum() < CFG['TOPK'] + 1:
            return naive, np.nan
        ai = a[mg] / (a[mg].sum() + 1e-12)
        within = np.sort(ai)[-CFG['TOPK']:].sum()
        return naive, within

    def cv_auc(X, y):
        X = np.nan_to_num(np.asarray(X, np.float64))
        cv = StratifiedKFold(5, shuffle=True, random_state=CFG['SEED'])
        pr = cross_val_predict(LogisticRegression(max_iter=2000), X, y,
                               cv=cv, method='predict_proba')[:, 1]
        return roc_auc_score(y, pr)

    SAM_RESULT = {}
    for name, (paths, y) in SAMPLES.items():
        keep, masks = SAM_MASKS[name]
        yy = y[keep]
        phi, sn, sw, ar = [], [], [], []
        for i, m in zip(tqdm(keep, desc='diag ' + name), masks):
            rgb = np.array(Image.open(paths[i]).convert('RGB'))
            if max(rgb.shape[:2]) > 1024:
                s = 1024 / max(rgb.shape[:2]); rgb = cv2.resize(rgb, None, fx=s, fy=s)
            if m.shape != rgb.shape[:2]:
                m = cv2.resize(m, (rgb.shape[1], rgb.shape[0]), interpolation=cv2.INTER_NEAREST)
            phi.append(shape_desc(m)); ar.append(m.mean())
            n_, w_ = attn_scores(rgb, m); sn.append(n_); sw.append(w_)
        sn, sw, ar, yy = map(np.asarray, (sn, sw, ar, yy))
        ok = ~np.isnan(sw)
        SAM_RESULT[name] = dict(
            n=int(len(yy)),
            ratio=float(ar[yy == 0].mean() / (ar[yy == 1].mean() + 1e-9)),
            shape=float(cv_auc(phi, yy)),
            naive=float(roc_auc_score(yy, sn)),
            within=float(roc_auc_score(yy[ok], sw[ok])),
            median_area=float(np.median(ar)))
        r = SAM_RESULT[name]; r['delta'] = r['naive'] - r['within']
        print('  %-20s ratio %.2f | shape %.3f | naive %.3f | within %.3f | d %+.3f | area %.3f'
              % (name, r['ratio'], r['shape'], r['naive'], r['within'], r['delta'], r['median_area']))

    json.dump(SAM_RESULT, open(os.path.join(CFG['FIG'], 'fig4_sam_diagnostic.json'), 'w'), indent=2)
    print('\nsaved -> fig4_sam_diagnostic.json')

In [ ]:
# ============================================================
# CELL 5 - FIGURE 4, ROUTE B  (redrawn from SAM masks)
# ============================================================
if RUN_SAM:
    order = [k for k in ORDER if k in SAM_RESULT]
    short = {'TeaLeafDiseaseBD': 'TeaLeaf\nDiseaseBD', 'PlantDoc': 'PlantDoc\n(field)',
             'PlantVillage': 'Plant\nVillage', 'RiceLeafDiseaseBD': 'RiceLeaf\nDiseaseBD',
             'RoCoLe': 'RoCoLe'}
    cols = [OURS if k == 'TeaLeafDiseaseBD' else EXT for k in order]
    x = np.arange(len(order))

    fig, ax = plt.subplots(1, 4, figsize=(13.5, 3.4))
    spec = [('a', 'ratio', 'Leaf-area ratio\n(healthy / diseased)', None, '%.2f'),
            ('b', 'shape', 'C1: leaf geometry alone\n(AUC)', 0.70, '%.2f'),
            ('c', 'delta', 'C2: attention collapse\nrestricted to the leaf', 0.15, '%+.2f'),
            ('d', 'median_area', 'Median segmented area\n(SAM, fraction of frame)', None, '%.2f')]
    for i, (tag, key, title, thr, fmt) in enumerate(spec):
        a = ax[i]
        v = [SAM_RESULT[k][key] for k in order]
        pad = (max(v) - min(min(v), 0)) * .06 + .02
        a.bar(x, v, color=cols, edgecolor='none', width=.68)
        for xi, vi in zip(x, v):
            a.text(xi, vi + pad * .35, fmt % vi, ha='center', fontsize=7.5,
                   fontweight='bold' if order[xi] == 'TeaLeafDiseaseBD' else 'normal')
        if thr is not None:
            a.axhline(thr, ls='--', lw=.9, color='#4A78B5')
            a.text(len(order) - .55, thr, ' confound\n threshold', fontsize=6.5,
                   color='#4A78B5', va='bottom', ha='right')
        if key == 'ratio':
            a.axhline(1.0, ls=':', lw=.9, color='#888')
        a.set_ylim(min(0, min(v) - pad), max(v) + pad * 2.2)
        a.set_xticks(x); a.set_xticklabels([short[k] for k in order], fontsize=6.5)
        a.set_title(title, fontsize=8.5)
        a.text(-.16, 1.14, tag, transform=a.transAxes, fontsize=12, fontweight='bold')

    fig.suptitle('The confound does not generalise (SAM segmentation)',
                 fontsize=10.5, y=1.04)
    fig.tight_layout()
    p = os.path.join(CFG['FIG'], 'fig4_routeB_sam.png')
    fig.savefig(p, dpi=300, bbox_inches='tight'); plt.show()
    print('saved ->', p)
    print('\nUse this ONLY if panel d shows the masks are now finding leaves')
    print('(median area well below 1.0). Otherwise use Route A.')

In [ ]:
# ============================================================
# CELL 6 - attention-head check: [0,1] hardcoded vs [1,3] from UAHS
# ============================================================
# TEA_LEAF_FIGURES.ipynb sets sel_heads = [0, 1]. UAHS actually selects
# [1, 3] in all five seeds. If any published attention figure came from
# that notebook, it shows heads the method does not select.

import timm
if 'enc' not in dir():
    raise RuntimeError('Run Cell 4 first - it defines the patched DINO encoder.')

MEAN = np.array([0.485, 0.456, 0.406]); STD = np.array([0.229, 0.224, 0.225])

@torch.no_grad()
def head_maps(rgb):
    x = cv2.resize(rgb, (224, 224)).astype(np.float32) / 255.
    x = ((x - MEAN) / STD).transpose(2, 0, 1)[None]
    _ = enc(torch.tensor(x, dtype=torch.float32, device=device))
    return _ATTN['cls'][0].numpy().reshape(-1, CFG['GRID'], CFG['GRID'])

tea = sorted(glob.glob(os.path.join(CFG['LOCAL_DATA'], '*', '*.jpg')))
rng = np.random.default_rng(CFG['SEED'])
sel = [tea[i] for i in rng.choice(len(tea), 4, replace=False)]

fig, ax = plt.subplots(4, 3, figsize=(7.5, 9.5))
for r, p in enumerate(sel):
    rgb = np.array(Image.open(p).convert('RGB'))
    hm = head_maps(rgb)
    small = cv2.resize(rgb, (224, 224))
    ax[r, 0].imshow(small); ax[r, 0].set_ylabel(os.path.basename(os.path.dirname(p)), fontsize=7)
    for c, (heads, lab) in enumerate([([0, 1], 'hardcoded [0,1]'), ([1, 3], 'UAHS [1,3]')]):
        m = hm[heads].mean(0)
        m = cv2.resize(m, (224, 224))
        ax[r, c + 1].imshow(small); ax[r, c + 1].imshow(m, cmap='inferno', alpha=.6)
        if r == 0:
            ax[r, c + 1].set_title(lab, fontsize=9)
    if r == 0:
        ax[r, 0].set_title('image', fontsize=9)
for a in ax.ravel():
    a.set_xticks([]); a.set_yticks([])
fig.tight_layout()
p = os.path.join(CFG['FIG'], 'head_check_01_vs_13.png')
fig.savefig(p, dpi=200, bbox_inches='tight'); plt.show()
print('saved ->', p)
print('\nIf the two columns look materially different AND a published figure')
print('came from TEA_LEAF_FIGURES.ipynb, regenerate it with heads [1,3].')

In [ ]:
# ============================================================
# CELL 6 - attention-head check: [0,1] hardcoded vs [1,3] from UAHS
# ============================================================
# TEA_LEAF_FIGURES.ipynb sets sel_heads = [0, 1]. UAHS actually selects
# [1, 3] in all five seeds. If any published attention figure came from
# that notebook, it shows heads the method does not select.

import timm
if 'enc' not in dir():
    raise RuntimeError('Run Cell 4 first - it defines the patched DINO encoder.')

MEAN = np.array([0.485, 0.456, 0.406]); STD = np.array([0.229, 0.224, 0.225])

@torch.no_grad()
def head_maps(rgb):
    x = cv2.resize(rgb, (224, 224)).astype(np.float32) / 255.
    x = ((x - MEAN) / STD).transpose(2, 0, 1)[None]
    _ = enc(torch.tensor(x, dtype=torch.float32, device=device))
    return _ATTN['cls'][0].numpy().reshape(-1, CFG['GRID'], CFG['GRID'])

tea = sorted(glob.glob(os.path.join(CFG['LOCAL_DATA'], '*', '*.jpg')))
rng = np.random.default_rng(CFG['SEED'])
sel = [tea[i] for i in rng.choice(len(tea), 4, replace=False)]

fig, ax = plt.subplots(4, 3, figsize=(7.5, 9.5))
for r, p in enumerate(sel):
    rgb = np.array(Image.open(p).convert('RGB'))
    hm = head_maps(rgb)
    small = cv2.resize(rgb, (224, 224))
    ax[r, 0].imshow(small); ax[r, 0].set_ylabel(os.path.basename(os.path.dirname(p)), fontsize=7)
    for c, (heads, lab) in enumerate([([0, 1], 'hardcoded [0,1]'), ([1, 3], 'UAHS [1,3]')]):
        m = hm[heads].mean(0)
        m = cv2.resize(m, (224, 224))
        ax[r, c + 1].imshow(small); ax[r, c + 1].imshow(m, cmap='inferno', alpha=.6)
        if r == 0:
            ax[r, c + 1].set_title(lab, fontsize=9)
    if r == 0:
        ax[r, 0].set_title('image', fontsize=9)
for a in ax.ravel():
    a.set_xticks([]); a.set_yticks([])
fig.tight_layout()
p = os.path.join(CFG['FIG'], 'head_check_01_vs_13.png')
fig.savefig(p, dpi=200, bbox_inches='tight'); plt.show()
print('saved ->', p)
print('\nIf the two columns look materially different AND a published figure')
print('came from TEA_LEAF_FIGURES.ipynb, regenerate it with heads [1,3].')